# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
import pandas as pd
from agent import Agent

The system prompt below defines who the advisor is, its key capabilities (tools), the steps it follows,
how to make recommendations and example questions it should handle.

On every call the agent also adds a context message with the current date and time, the next 7 days, the location
and the home's device profile, so it can resolve "tomorrow" or "Wednesday" and personalize its advice.

In [2]:
## Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """You are the EcoHome Energy Advisor, an expert in residential energy optimization working for \
EcoHome, a smart-home energy company. You help customers with rooftop solar, electric vehicles, smart thermostats and \
smart appliances lower their electricity bill and carbon footprint with specific, data-driven recommendations.

## Key capabilities (your tools)
- get_weather_forecast: hourly weather, solar irradiance and estimated solar generation for up to 7 days, \
with a per-day summary and best solar window. Use it for any question about tomorrow, a weekday, "this week" \
or solar production.
- get_electricity_prices: time-of-use rates per hour for a date (off_peak, mid_peak, peak) and the rate of each \
period. Use it whenever timing or cost matters. Call it once per date you are recommending for. When the customer \
mentions a specific time, pass that hour (e.g. 7 PM -> hour=19) to get its exact rate.
- query_energy_usage: the customer's historical consumption by device (EV, HVAC, appliance) with hourly profile.
- query_solar_generation: the customer's historical solar production by day, hour and weather.
- get_recent_energy_summary: last N hours of usage vs. generation.
- search_energy_tips: the EcoHome knowledge base of energy-saving best practices (devices, EV charging, HVAC, \
smart home automation, solar integration, seasonal management, battery storage, time-of-use pricing).
- calculate_energy_savings: savings in kWh, USD, % and CO2. Use it for every savings figure you quote; to \
model shifting usage to a cheaper time, pass the old rate as price_per_kwh and the new rate as \
optimized_price_per_kwh.

## Steps
1. Work out what the customer is asking and which dates it refers to. Resolve relative dates ("tomorrow", \
"Wednesday", "this week") using the current date given in the context.
2. Gather data before answering. Never invent prices, forecasts, usage numbers or savings - get them from a \
tool. Typical combinations:
   - Scheduling a device (EV, dishwasher, laundry, pool pump): weather forecast + electricity prices, then \
pick the hours that are cheap and/or covered by solar.
   - Thermostat / HVAC: weather forecast (temperatures) + electricity prices + search_energy_tips.
   - "Based on my usage/history": query_energy_usage (and query_solar_generation if solar is relevant), \
using the last 30 days unless the customer says otherwise.
   - "How much can I save": get the real rates (and usage if needed), then calculate_energy_savings.
   - Thermostat questions: follow the day's "hvac_need" in the forecast. "cooling" -> cooling setpoint and \
pre-cooling; "heating" -> heating setpoint (about 68°F / 20°C), pre-heating and a setback during the peak, and \
do not recommend cooling at all; "none" -> say little HVAC is needed and suggest a wide setback.
3. Always call search_energy_tips for at least one relevant topic before giving your final answer, so advice \
is grounded in the knowledge base. This applies to every energy question, including savings and history questions.
4. Analyze the data and compare options (e.g. solar window vs. off-peak vs. peak) before choosing one.
5. If a tool returns an error or no data, say so briefly, fall back to sensible general guidance and state \
your assumptions. Do not retry the same failing call more than once.

## Recommendations
- Prefer free solar energy during the best solar window on sunny or partly cloudy days; otherwise prefer \
off-peak hours (22:00-06:00). Avoid peak hours (weekdays 16:00-21:00) for flexible loads.
- Use the rate of the exact hour or period being compared, never an average. A time the customer names \
(e.g. 7 PM) is priced at that hour's rate. When the current time of a flexible load is not given, assume it runs \
in the evening peak (16:00-21:00) and state that assumption.
- Sanity-check figures against typical household values (an EV uses ~8-12 kWh per day, a dishwasher ~1.5 kWh \
per load); if data looks unusual, point it out instead of extrapolating blindly.
- For thermostats: around 78°F (25-26°C) when cooling and 68°F (20°C) when heating; pre-cool or pre-heat \
before peak prices and let the temperature drift during the peak.
- Lead with a direct answer: the recommended time window (with date, 24h times), setting or top actions.
- Support it with the numbers you used: rates in $/kWh, forecast conditions, expected solar kWh, usage figures.
- Quantify savings in dollars (per occurrence and per year) and, when relevant, kWh and CO2 avoided.
- Cite knowledge base tips by source, e.g. "(EcoHome tip: tip_ev_charging.txt)". Only cite file names that \
search_energy_tips actually returned in this conversation - never make up a source.
- Name price periods exactly as get_electricity_prices labels them (off_peak 22:00-06:00, mid_peak, peak). \
Daytime solar hours are usually mid_peak, not off-peak.
- For heating advice, talk about pre-heating and setbacks; only mention pre-cooling when cooling is needed.
- Keep it concise and practical: short paragraphs or bullet points.
- Only answer questions about home energy. Politely decline unrelated requests without calling tools.

## Example questions you should handle
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?" -> forecast + prices, \
recommend a window, compare cost vs. peak, cite an EV tip.
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?" -> forecast for \
Wednesday + prices + HVAC tips, give a setpoint and a pre-cooling plan.
- "Suggest three ways I can reduce energy use based on my usage history." -> usage history + tips, three actions \
tied to the biggest loads, with estimated savings.
- "How much can I save by running my dishwasher during off-peak hours?" -> prices + calculate_energy_savings.
- "What's the best time to run my pool pump this week based on the weather forecast?" -> 7-day forecast + prices, \
a window per day.
"""

print(ECOHOME_SYSTEM_PROMPT)

You are the EcoHome Energy Advisor, an expert in residential energy optimization working for EcoHome, a smart-home energy company. You help customers with rooftop solar, electric vehicles, smart thermostats and smart appliances lower their electricity bill and carbon footprint with specific, data-driven recommendations.

## Key capabilities (your tools)
- get_weather_forecast: hourly weather, solar irradiance and estimated solar generation for up to 7 days, with a per-day summary and best solar window. Use it for any question about tomorrow, a weekday, "this week" or solar production.
- get_electricity_prices: time-of-use rates per hour for a date (off_peak, mid_peak, peak) and the rate of each period. Use it whenever timing or cost matters. Call it once per date you are recommending for. When the customer mentions a specific time, pass that hour (e.g. 7 PM -> hour=19) to get its exact rate.
- query_energy_usage: the customer's historical consumption by device (EV, HVAC, appliance) wit

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)
print("Tools:", ecohome_agent.get_agent_tools())
print()
print(ecohome_agent.build_context("Location: San Francisco, CA"))

Tools: ['get_weather_forecast', 'get_electricity_prices', 'query_energy_usage', 'query_solar_generation', 'get_recent_energy_summary', 'search_energy_tips', 'calculate_energy_savings']

Current date and time: Wednesday, 2026-10-28 01:40
Tomorrow is Thursday, 2026-10-29.
Upcoming days: Wed 2026-10-28, Thu 2026-10-29, Fri 2026-10-30, Sat 2026-10-31, Sun 2026-11-01, Mon 2026-11-02, Tue 2026-11-03
Home profile: rooftop solar (about 5 kW peak), Tesla Model 3 EV, central AC (HVAC), dishwasher, washing machine and dryer. Usage history covers roughly the last 30 days.
Location: San Francisco, CA


In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow (Thursday, October 29), I recommend the following time window:

### Recommended Charging Window:
- **Time:** 10:00 AM to 3:00 PM
- **Solar Generation:** This period coincides with the best solar window, where your solar panels will generate the most energy.

### Cost Analysis:
- **Electricity Rates:**
  - **Off-Peak (22:00 - 06:00):** $0.092/kWh
  - **Mid-Peak (06:00 - 16:00):** $0.1533/kWh
  - **Peak (16:00 - 21:00):** $0.2759/kWh

### Solar Generation Forecast:
- **Estimated Solar Generation:**
  - **10:00 AM:** 4.43 kWh
  - **11:00 AM:** 5.11 kWh
  - **12:00 PM:** 4.75 kWh
  - **1:00 PM:** 4.38 kWh
  - **2:00 PM:** 4.46 kWh
  - **3:00 PM:** 3.49 kWh

This means that during this window, you can charge your EV using the solar energy generated, significantly reducing your electricity costs.

### Additional Tip:
Consider scheduling your EV charging to start just before the solar generation peaks (around 

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    for call in getattr(msg, "tool_calls", None) or []:
        print("-", call["name"], call["args"])

TOOLS:
- get_weather_forecast {'location': 'San Francisco, CA', 'days': 1}
- get_electricity_prices {'date': '2026-10-29'}


### Context awareness and error handling
A follow-up question that reuses the same `thread_id` builds on the previous answer.
Invalid input returns a friendly message in the same response shape instead of raising.

In [7]:
thread = "demo-conversation"
first = ecohome_agent.invoke(
    question="How much energy did my EV use over the last 7 days?",
    context="Location: San Francisco, CA",
    thread_id=thread,
)
print(first["messages"][-1].content)
print("\n" + "=" * 60 + "\n")
follow_up = ecohome_agent.invoke(
    question="And how much would I save per year if I moved half of that to off-peak hours?",
    context="Location: San Francisco, CA",
    thread_id=thread,
)
print(follow_up["messages"][-1].content)

Over the last 7 days, your Tesla Model 3 EV consumed a total of **74.07 kWh**. This averages to about **10.58 kWh per day**. The total cost for this energy usage was approximately **$15.61**. 

If you have any further questions or need assistance with optimizing your EV charging, feel free to ask!




If you shift half of your EV's weekly usage (approximately 37.035 kWh) to off-peak hours, you could save about **$2,445.37 per year**. This represents a savings of approximately **66.7%** compared to charging during peak hours.

### Breakdown:
- **Current Cost (Peak)**: ~$10.05 for half of the weekly usage.
- **Optimized Cost (Off-Peak)**: ~$3.35 for the same usage.
- **Annual Savings**: **$2,445.37**

This strategy not only reduces your electricity bill but also helps in managing your energy consumption more efficiently. If you have any further questions or need assistance with implementation, feel free to ask!


In [8]:
# Error handling: an empty question doesn't reach the model
print(ecohome_agent.invoke(question="   ")["messages"][-1].content)

Please ask a question about your home energy use.


## 2. Define Test Cases

In [9]:
# Define comprehensive test cases for the Energy Advisor
# 12 test cases covering:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations
# - Historical usage analysis, battery storage and an off-topic request

In [10]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "thermostat_price_spike",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "A specific thermostat setpoint for the afternoon peak that fits Wednesday's forecast: around "
                             "78-80F / 25-27C with pre-cooling before the peak if it is hot, or around 66-68F / 19-20C with "
                             "pre-heating and a setback during the peak if it is cool. It should state the peak prices and "
                             "the forecast temperature for Wednesday",
    },
    {
        "id": "usage_history_reduction",
        "question": "Suggest three ways I can reduce energy use based on my usage history.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "Exactly three concrete suggestions tied to the customer's actual usage data (e.g. EV is the largest load, "
                             "usage concentrated in peak hours), with kWh or dollar figures and tips from the knowledge base",
    },
    {
        "id": "dishwasher_off_peak_savings",
        "question": "How much can I save by running my dishwasher during off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "A dollar savings estimate per cycle and per year based on the peak vs. off-peak rates, "
                             "the recommended off-peak time window, and any assumptions such as kWh per cycle",
    },
    {
        "id": "pool_pump_weekly",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Recommended pool pump run windows for the coming days that use sunny/solar hours where possible and "
                             "off-peak hours on cloudy days, referencing the forecast conditions and electricity rates",
    },
    {
        "id": "solar_self_consumption",
        "question": "Look at my solar production over the last two weeks. How can I use more of my own solar power?",
        "expected_tools": ["query_solar_generation", "search_energy_tips"],
        "expected_response": "A summary of recent solar generation (totals, peak production hours, weather impact) and specific "
                             "load-shifting suggestions to use more solar during the midday window",
    },
    {
        "id": "recent_energy_balance",
        "question": "How did my energy consumption compare to my solar generation over the last 24 hours?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "Total consumption and generation for the last 24 hours, the net grid usage or solar coverage, "
                             "the device breakdown and a short interpretation",
    },
    {
        "id": "ev_shift_annual_savings",
        "question": "If I move my daily EV charging from 7 PM to off-peak hours overnight, how much will I save per year?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Daily EV energy use from the history, the 7 PM peak rate vs. the off-peak rate, and calculated "
                             "per-day and annual dollar savings",
    },
    {
        "id": "hvac_weekly_cost",
        "question": "What did my air conditioning cost me over the past week, and how can I lower that cost?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The HVAC consumption and cost for the past 7 days from the database, followed by specific "
                             "HVAC cost reduction tips such as thermostat setpoints, pre-cooling and maintenance",
    },
    {
        "id": "home_battery_strategy",
        "question": "Would a home battery help me save money with my electricity rates? How should I run it?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "An explanation using the actual peak and off-peak rates of how a battery would charge from solar or "
                             "off-peak power and discharge during the peak, with an estimate of daily or annual savings",
    },
    {
        "id": "laundry_tomorrow",
        "question": "When is the best time tomorrow to run my washing machine and dryer?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "A specific time window for tomorrow based on solar forecast and electricity rates, "
                             "avoiding peak hours, with tips like cold water or full loads",
    },
    {
        "id": "off_topic_request",
        "question": "Can you recommend a good pizza place near me?",
        "expected_tools": [],
        "expected_response": "A polite refusal explaining the advisor only helps with home energy topics, without calling any tools",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [11]:
CONTEXT = "Location: San Francisco, CA"

In [12]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation, including a log of the agent's decisions and tool usage

from langchain_core.messages import AIMessage, ToolMessage


def extract_tool_log(messages):
    """Each tool the agent decided to call, with its arguments and a preview of the result."""
    results = {m.tool_call_id: m for m in messages if isinstance(m, ToolMessage)}
    log = []
    for msg in messages:
        if isinstance(msg, AIMessage):
            for call in msg.tool_calls or []:
                result = results.get(call["id"])
                log.append({
                    "tool": call["name"],
                    "args": call["args"],
                    "result_preview": str(result.content)[:300] if result else None,
                    "error": bool(result is not None and ("error" in str(result.content)[:30].lower())),
                })
    return log


print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)

    try:
        start = datetime.now()
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        tool_log = extract_tool_log(response["messages"])

        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'final_response': response["messages"][-1].content,
            'tools_used': [t["tool"] for t in tool_log],
            'tool_log': tool_log,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'duration_s': round((datetime.now() - start).total_seconds(), 1),
            'timestamp': datetime.now().isoformat()
        }
        if "error" in response:
            result["error"] = response["error"]
        test_results.append(result)

        for t in tool_log:
            print(f"  tool: {t['tool']}({t['args']})")
        print(f"  answer: {result['final_response'][:200]}...")

    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'final_response': f"Error: {str(e)}",
            'tools_used': [],
            'tool_log': [],
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")

=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------


  tool: get_weather_forecast({'location': 'San Francisco, CA', 'days': 1})
  tool: get_electricity_prices({'date': '2026-10-29'})
  answer: To minimize costs and maximize solar power when charging your electric car tomorrow (Thursday, October 29), I recommend the following time window:

### Recommended Charging Window:
- **Time:** 10:00 A...

Test 2: thermostat_price_spike
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------


  tool: get_weather_forecast({'location': 'San Francisco, CA', 'days': 1})
  tool: get_electricity_prices({'date': '2026-10-28'})
  tool: search_energy_tips({'query': 'thermostat settings', 'max_results': 5})
  answer: For Wednesday afternoon (October 28), if electricity prices spike, I recommend setting your thermostat to **68°F (20°C)**. This is the optimal temperature for heating when you're awake, as suggested b...

Test 3: usage_history_reduction
Question: Suggest three ways I can reduce energy use based on my usage history.
--------------------------------------------------


  tool: query_energy_usage({'start_date': '2026-09-28', 'end_date': '2026-10-28'})
  tool: query_solar_generation({'start_date': '2026-09-28', 'end_date': '2026-10-28'})
  tool: search_energy_tips({'query': 'reduce energy use'})
  answer: Here are three effective ways to reduce your energy use based on your recent usage history:

1. **Optimize HVAC Usage**:
   - Your HVAC system accounts for about 58% of your total energy consumption. ...

Test 4: dishwasher_off_peak_savings
Question: How much can I save by running my dishwasher during off-peak hours?
--------------------------------------------------


  tool: get_electricity_prices({'date': '2026-10-29'})
  tool: query_energy_usage({'start_date': '2026-09-29', 'end_date': '2026-10-29', 'device_type': 'appliance'})
  tool: calculate_energy_savings({'device_type': 'Dishwasher', 'current_usage_kwh': 1.5, 'optimized_usage_kwh': 1.5, 'price_per_kwh': 0.2759, 'optimized_price_per_kwh': 0.092, 'occurrences_per_year': 365})
  answer: By running your dishwasher during off-peak hours, you can save significantly on your electricity costs. Here's a breakdown of the savings:

- **Current Cost (Peak Hour)**: Approximately **$0.41** per ...

Test 5: pool_pump_weekly
Question: What's the best time to run my pool pump this week based on the weather forecast?
--------------------------------------------------


...

  tool: get_electricity_prices({'date': '2026-10-29', 'hour': 19})
  tool: get_electricity_prices({'date': '2026-10-29'})
  tool: query_energy_usage({'start_date': '2026-09-29', 'end_date': '2026-10-29', 'device_type': 'EV'})
  tool: calculate_energy_savings({'device_type': 'EV', 'current_usage_kwh': 9.08, 'optimized_usage_kwh': 9.08, 'price_per_kwh': 0.2759, 'optimized_price_per_kwh': 0.092, 'occurrences_per_year': 365})
  answer: By moving your daily EV charging from 7 PM (peak hours) to off-peak hours overnight, you can save approximately **$609.48 per year**. 

### Breakdown:
- **Current Cost (Peak)**: $2.51 per day
- **Opti...

Test 9: hvac_weekly_cost
Question: What did my air conditioning cost me over the past week, and how can I lower that cost?
--------------------------------------------------


  tool: query_energy_usage({'start_date': '2026-10-21', 'end_date': '2026-10-28', 'device_type': 'HVAC'})
  tool: search_energy_tips({'query': 'HVAC', 'max_results': 5})
  answer: Over the past week (October 21-28, 2026), your air conditioning (HVAC) cost you approximately **$19.98**, consuming about **126.03 kWh** of electricity. This averages to about **18.0 kWh per day**.

#...

Test 10: home_battery_strategy
Question: Would a home battery help me save money with my electricity rates? How should I run it?
--------------------------------------------------


  tool: get_electricity_prices({})
  tool: query_energy_usage({'start_date': '2026-09-28', 'end_date': '2026-10-28'})
  tool: query_solar_generation({'start_date': '2026-09-28', 'end_date': '2026-10-28'})
  answer: A home battery can indeed help you save money on your electricity bills, especially given your current electricity rates and solar generation profile. Here's how it can work for you:

### Current Elec...

Test 11: laundry_tomorrow
Question: When is the best time tomorrow to run my washing machine and dryer?
--------------------------------------------------


  tool: get_weather_forecast({'location': 'San Francisco, CA', 'days': 1})
  tool: get_electricity_prices({'date': '2026-10-29'})
  answer: To run your washing machine and dryer tomorrow (Thursday, October 29), the best time is during the **off-peak hours** from **10:00 PM to 6:00 AM**. Here's a breakdown:

### Recommended Time Window:
- ...

Test 12: off_topic_request
Question: Can you recommend a good pizza place near me?
--------------------------------------------------


  answer: I'm here to help with energy optimization and home energy questions. If you have any inquiries about your energy usage, solar power, or electric vehicles, feel free to ask!...

Completed 12 tests


In [13]:
# Log of each test: decisions (tool calls) and final recommendation
for result in test_results:
    print(f"### {result['test_id']}: {result['question']}")
    print("Tools:", ", ".join(result["tools_used"]) or "none")
    print(result["final_response"])
    print()

### ev_charging_1: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
Tools: get_weather_forecast, get_electricity_prices
To minimize costs and maximize solar power when charging your electric car tomorrow (Thursday, October 29), I recommend the following time window:

### Recommended Charging Window:
- **Time:** 10:00 AM to 3:00 PM
- **Solar Generation:** This period coincides with the best solar window, where your solar panels are expected to generate the most energy.

### Solar Generation and Electricity Prices:
- **Estimated Solar Generation:** 
  - 10:00 AM: 4.43 kWh
  - 11:00 AM: 5.11 kWh
  - 12:00 PM: 4.75 kWh
  - 1:00 PM: 4.38 kWh
  - 2:00 PM: 4.46 kWh
  - 3:00 PM: 3.49 kWh
- **Electricity Rates:**
  - **Mid-Peak Rate (10:00 AM - 3:00 PM):** $0.1533 per kWh

### Summary:
Charging during this window will allow you to utilize the solar energy generated by your rooftop system, reducing your reliance on grid electricity and minimizing costs. 



## 4. Evaluate Responses

Two evaluators are used:
- **Response quality** (`evaluate_response`, LLM-as-judge): an evaluator model scores each final answer from 1-5 on
  **accuracy, relevance, completeness, usefulness** and **reasoning** against the expected response, and writes feedback
  (strengths, weaknesses, suggestions). To judge accuracy it is also given the tool outputs the agent saw, so it can
  check that numbers weren't made up.
- **Tool usage** (`evaluate_tool_usage`, deterministic): **tool appropriateness** (share of the tools called that were
  expected for the task) and **tool completeness** (share of the expected tools that were called), with feedback
  listing missing, unexpected and failed tool calls.

`generate_evaluation_report()` computes the metrics and returns a structured report;
`display_evaluation_report()` presents it.

In [14]:
import os
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

CRITERIA = ["accuracy", "relevance", "completeness", "usefulness", "reasoning"]


class ResponseScore(BaseModel):
    accuracy: int = Field(description="1-5: numbers and facts are correct and consistent with the tool outputs, nothing invented")
    relevance: int = Field(description="1-5: the response directly addresses the user's question")
    completeness: int = Field(description="1-5: covers everything in the expected response")
    usefulness: int = Field(description="1-5: the customer could act on it right away (specific times, settings, dollar amounts)")
    reasoning: int = Field(description="1-5: recommendations are clearly explained and justified with data")
    strengths: str = Field(description="What the response does well")
    weaknesses: str = Field(description="What is missing, wrong or unclear")
    suggestions: str = Field(description="How the response could be improved")


# A stronger model than the agent's gpt-4o-mini acts as the judge, so the agent isn't grading itself
judge_llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY"),
).with_structured_output(ResponseScore, method="function_calling")


def get_messages(response):
    return response["messages"] if isinstance(response, dict) else []


def get_tool_context(messages, max_chars_per_tool=3000):
    """Condensed tool outputs so the judge can verify the numbers in the answer."""
    parts = [f"[{m.name}] {str(m.content)[:max_chars_per_tool]}" for m in messages if isinstance(m, ToolMessage)]
    return "\n".join(parts) or "No tools were called."

In [15]:
# Response evaluator
def evaluate_response(question, final_response, expected_response, tool_context=None):
    """Evaluate a single response against expected response"""
    prompt = f"""You are a strict evaluator of an AI home energy advisor. Score the response on each criterion from 1 (poor)
to 5 (excellent) and give specific feedback.

Scoring rules:
- Check every number in the response against the tool outputs. A rate that doesn't match the hour or period being
  discussed (e.g. using an average or mid-peak rate for "7 PM", which is peak), a miscalculation, or a figure
  that is not in the tool outputs lowers accuracy to 3 or less.
- Advice that contradicts the data (e.g. pre-cooling when the forecast is cool, charging at a time that isn't cheap)
  lowers accuracy and reasoning.
- Implausible results presented without comment (e.g. thousands of dollars per year for one appliance) lower
  reasoning and usefulness.
- Give 5 only when the criterion is fully met with no issues. Always name at least one weakness unless the
  response is flawless.

USER QUESTION:
{question}

EXPECTED RESPONSE (what a good answer should contain):
{expected_response}

TOOL OUTPUTS THE ADVISOR SAW (ground truth for accuracy):
{tool_context or "Not provided."}

ADVISOR RESPONSE:
{final_response}
"""
    try:
        result = judge_llm.invoke(prompt).model_dump()
    except Exception as e:
        result = {c: 0 for c in CRITERIA}
        result.update(strengths="", weaknesses=f"Evaluation failed: {e}", suggestions="")
    for c in CRITERIA:
        result[c] = max(0, min(5, int(result[c])))
    result["overall"] = round(sum(result[c] for c in CRITERIA) / len(CRITERIA), 2)
    result["feedback"] = (
        f"Strengths: {result['strengths']} Weaknesses: {result['weaknesses']} Suggestions: {result['suggestions']}"
    )
    return result

In [16]:
# Tool usage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    tool_calls = [
        call["name"]
        for msg in messages if isinstance(msg, AIMessage)
        for call in (msg.tool_calls or [])
    ]
    failed = sorted({
        m.name for m in messages
        if isinstance(m, ToolMessage) and "error" in str(m.content)[:30].lower()
    })
    used = set(tool_calls)
    expected = set(expected_tools)
    missing = sorted(expected - used)
    unexpected = sorted(used - expected)

    if expected:
        # Tool completeness: were all necessary tools used?
        completeness = len(used & expected) / len(expected)
        # Tool appropriateness: were the tools that were selected the right ones for the task?
        appropriateness = len(used & expected) / len(used) if used else 0.0
    else:
        # No tools expected (e.g. off-topic): full marks only if none were called
        completeness = appropriateness = 1.0 if not used else 0.0

    feedback = []
    if not expected and not used:
        feedback.append("Correctly answered without calling any tools.")
    elif not missing:
        feedback.append(f"All necessary tools were used ({', '.join(sorted(expected))}).")
    else:
        feedback.append(f"Missing necessary tools: {', '.join(missing)}.")
    if unexpected:
        feedback.append(f"Additional tools called: {', '.join(unexpected)} - acceptable if they supported the answer.")
    if failed:
        feedback.append(f"Tool calls that returned errors: {', '.join(failed)}.")
    repeats = sorted({t for t in tool_calls if tool_calls.count(t) > 1})
    if repeats:
        feedback.append(f"Called more than once: {', '.join(repeats)}.")

    return {
        "tools_used": sorted(used),
        "total_tool_calls": len(tool_calls),
        "missing_tools": missing,
        "unexpected_tools": unexpected,
        "failed_tools": failed,
        "tool_appropriateness": round(appropriateness, 2),
        "tool_completeness": round(completeness, 2),
        # Completeness weighs more: extra supporting tools (e.g. a tip search) are usually fine
        "tool_score": round(0.7 * completeness + 0.3 * appropriateness, 2),
        "feedback": " ".join(feedback),
    }

In [17]:
# Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results=None, pass_threshold=3.5):
    """Evaluate every test result and return a structured report (dict)."""
    results = results if results is not None else test_results
    rows = []
    for r in results:
        messages = get_messages(r["response"])
        tool_eval = evaluate_tool_usage(messages, r["expected_tools"])
        response_eval = evaluate_response(
            r["question"], r["final_response"], r["expected_response"], get_tool_context(messages)
        )
        rows.append({
            "test_id": r["test_id"],
            **{c: response_eval[c] for c in CRITERIA},
            "response_score": response_eval["overall"],
            "tool_appropriateness": tool_eval["tool_appropriateness"],
            "tool_completeness": tool_eval["tool_completeness"],
            "tool_score": tool_eval["tool_score"],
            "tools_used": ", ".join(tool_eval["tools_used"]),
            "missing_tools": ", ".join(tool_eval["missing_tools"]),
            "errored": "error" in r,
            "response_feedback": response_eval["feedback"],
            "tool_feedback": tool_eval["feedback"],
        })

    df = pd.DataFrame(rows)
    # Combined score on a 0-1 scale: 75% response quality, 25% tool usage
    df["combined_score"] = (0.75 * df["response_score"] / 5 + 0.25 * df["tool_score"]).round(2)
    df["passed"] = (df["response_score"] >= pass_threshold) & (df["tool_completeness"] >= 0.5) & ~df["errored"]

    criteria_means = df[CRITERIA].mean().round(2)
    missing_counts = pd.Series(
        [t for m in df["missing_tools"] for t in m.split(", ") if t], dtype=str
    ).value_counts()

    strengths = [f"Strongest criterion: {criteria_means.idxmax()} ({criteria_means.max():.2f}/5)"]
    best = df.sort_values("combined_score", ascending=False).head(3)
    strengths.append(f"Best test cases: {', '.join(best['test_id'])}")
    if df["tool_completeness"].mean() >= 0.9:
        strengths.append("Consistently uses all the necessary tools")
    if df["tool_appropriateness"].mean() >= 0.8:
        strengths.append("Tool selection is focused, with few unnecessary calls")
    if not df["errored"].any():
        strengths.append("No runtime errors across the test set")

    weaknesses = [f"Weakest criterion: {criteria_means.idxmin()} ({criteria_means.min():.2f}/5)"]
    for _, row in df[~df["passed"]].iterrows():
        weaknesses.append(
            f"{row['test_id']} failed (response {row['response_score']}/5, tool completeness "
            f"{row['tool_completeness']}): {row['tool_feedback']}"
        )
    if not missing_counts.empty:
        weaknesses.append("Most often missed tools: " + ", ".join(f"{t} ({n}x)" for t, n in missing_counts.items()))

    recommendations = []
    if criteria_means["accuracy"] < 4:
        recommendations.append("Tighten the prompt so every figure is quoted from a tool output, and use calculate_energy_savings for all savings.")
    if criteria_means["completeness"] < 4:
        recommendations.append("Add an answer checklist to the prompt (time window, rates, solar, savings, tip citation).")
    if criteria_means["usefulness"] < 4:
        recommendations.append("Ask for more concrete actions: exact times, setpoints and dollar amounts.")
    if criteria_means["reasoning"] < 4:
        recommendations.append("Ask the agent to explain why a window was chosen by comparing it with the alternatives.")
    if not missing_counts.empty:
        recommendations.append(f"Make the prompt more explicit about when to call: {', '.join(missing_counts.index)}.")
    if df["tool_appropriateness"].mean() < 0.6:
        recommendations.append("Reduce unnecessary tool calls to cut latency and cost.")
    if not recommendations:
        recommendations.append("Scores are strong; expand the test set with harder multi-device and multi-day scenarios.")

    return {
        "summary": {
            "test_cases": len(df),
            "passed": int(df["passed"].sum()),
            "pass_rate_pct": round(100 * df["passed"].mean(), 1),
            "errors": int(df["errored"].sum()),
            "avg_response_score": round(df["response_score"].mean(), 2),
            "avg_tool_appropriateness": round(df["tool_appropriateness"].mean(), 2),
            "avg_tool_completeness": round(df["tool_completeness"].mean(), 2),
            "avg_combined_score": round(df["combined_score"].mean(), 2),
        },
        "criteria_scores": criteria_means.to_dict(),
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
        "details": df,
    }

In [18]:
def display_evaluation_report(report):
    """Print a structured view of the report produced by generate_evaluation_report()."""
    s = report["summary"]
    print("=" * 70)
    print("ECOHOME ENERGY ADVISOR - EVALUATION REPORT")
    print("=" * 70)

    print("\n1. OVERALL METRICS")
    print(f"   Test cases:               {s['test_cases']}")
    print(f"   Passed:                   {s['passed']}/{s['test_cases']} ({s['pass_rate_pct']:.0f}%)")
    print(f"   Errors:                   {s['errors']}")
    print(f"   Avg response score:       {s['avg_response_score']:.2f} / 5")
    print(f"   Avg tool appropriateness: {s['avg_tool_appropriateness']:.2f}")
    print(f"   Avg tool completeness:    {s['avg_tool_completeness']:.2f}")
    print(f"   Avg combined score:       {s['avg_combined_score']:.2f} / 1")

    print("\n2. RESPONSE QUALITY BY CRITERION")
    for criterion, score in report["criteria_scores"].items():
        print(f"   {criterion:<13} {score:.2f}  {'#' * round(score * 4)}")

    for number, title, key in [(3, "STRENGTHS", "strengths"), (4, "WEAKNESSES", "weaknesses"),
                               (5, "RECOMMENDATIONS", "recommendations")]:
        print(f"\n{number}. {title}")
        for item in report[key]:
            print(f"   - {item}")

    print("\n6. RESULTS PER TEST CASE")
    display(report["details"][["test_id", *CRITERIA, "response_score", "tool_appropriateness",
                               "tool_completeness", "combined_score", "passed"]])

    print("\n7. FEEDBACK PER TEST CASE")
    for _, row in report["details"].iterrows():
        print(f"\n[{row['test_id']}] {'PASS' if row['passed'] else 'FAIL'}")
        print(f"   Response: {row['response_feedback']}")
        print(f"   Tools:    {row['tool_feedback']}")

In [19]:
evaluation_report = generate_evaluation_report(test_results)
display_evaluation_report(evaluation_report)

ECOHOME ENERGY ADVISOR - EVALUATION REPORT

1. OVERALL METRICS
   Test cases:               12
   Passed:                   11/12 (92%)
   Errors:                   0
   Avg response score:       4.72 / 5
   Avg tool appropriateness: 0.82
   Avg tool completeness:    0.96
   Avg combined score:       0.94 / 1

2. RESPONSE QUALITY BY CRITERION
   accuracy      4.58  ##################
   relevance     4.92  ####################
   completeness  4.75  ###################
   usefulness    4.75  ###################
   reasoning     4.58  ##################

3. STRENGTHS
   - Strongest criterion: relevance (4.92/5)
   - Best test cases: ev_shift_annual_savings, hvac_weekly_cost, laundry_tomorrow
   - Consistently uses all the necessary tools
   - Tool selection is focused, with few unnecessary calls
   - No runtime errors across the test set

4. WEAKNESSES
   - Weakest criterion: accuracy (4.58/5)
   - pool_pump_weekly failed (response 3.2/5, tool completeness 1.0): All necessary tools were

,test_id,accuracy,relevance,completeness,usefulness,reasoning,response_score,tool_appropriateness,tool_completeness,combined_score,passed
0,ev_charging_1,4,5,4,4,4,4.2,1.00,1.0,0.88,True
1,thermostat_price_spike,4,5,5,5,4,4.6,1.00,1.0,0.94,True
2,usage_history_reduction,5,5,5,5,5,5.0,0.67,1.0,0.98,True
3,dishwasher_off_peak_savings,5,5,5,5,5,5.0,0.67,1.0,0.98,True
4,pool_pump_weekly,3,4,3,3,3,3.2,1.00,1.0,0.73,False
5,solar_self_consumption,4,5,5,5,4,4.6,0.67,1.0,0.91,True
6,recent_energy_balance,5,5,5,5,5,5.0,0.50,1.0,0.96,True
7,ev_shift_annual_savings,5,5,5,5,5,5.0,1.00,1.0,1.00,True
8,hvac_weekly_cost,5,5,5,5,5,5.0,1.00,1.0,1.00,True
9,home_battery_strategy,5,5,5,5,5,5.0,0.33,0.5,0.86,True



7. FEEDBACK PER TEST CASE

[ev_charging_1] PASS
   Response: Strengths: The response provides a clear recommendation for the charging window, aligning with the best solar generation period. It also includes specific solar generation estimates and electricity rates, which are helpful for decision-making. Weaknesses: The response does not address the fact that charging during the recommended window incurs mid-peak electricity rates, which are higher than off-peak rates. It could also mention the potential cost savings of charging during off-peak hours, even if solar generation is lower. Suggestions: To improve the response, it should compare the cost of charging during the recommended solar window with the cost of charging during off-peak hours. This would provide a more comprehensive analysis of cost versus solar utilization. Additionally, it could suggest a hybrid approach, such as starting charging during off-peak hours and continuing into the solar window.
   Tools:    All necessary